# ME324 - Lab 4: Backpropagation - add `backward()` and train a net by hand

**Lecture 4 - "Backpropagation" - 6 August 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)  (In Colab: **File > Upload notebook**, or open from your course repo.)

### Today's goal
Yesterday you built the **forward pass** of a tiny autograd engine - the `Value`
class that records a computation graph. Today you add the other half: **`backward()`**,
the algorithm that computes every gradient by walking the graph in reverse. Then
you'll use it to **train a real (tiny) neural network by hand** and watch the loss fall.

By the end you will have:

- added **`backward()`** to `Value` (reverse-mode autodiff = backpropagation);
- verified your gradients **two ways**: against the lecture, and with a finite-difference check;
- built a `Neuron` / `Layer` / `MLP` on top of `Value`;
- **trained it** with the forward -> loss -> backward -> update loop, and plotted the loss.

> **This lab completes the fundamental engine of deep learning.** The exact `Value` class you complete today is
> reused, *unchanged*, in the **Lab 11 capstone** - where you'll watch a GPT run on it.

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**And please use an LLM.** ChatGPT, Claude, Gemini, DeepSeek — whichever you like.
*"In Python, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** adding `backward()`, the finite-difference grad-check, and training the tiny net.
- **Stretch / take-home — skip if short on time:** the extra `Neuron`/`Layer`/`MLP` niceties — read if time allows.

_Most of the code is written for you; the `# TODO` cells are the parts you write. Worked answers are in the **Solutions** section at the bottom._

In [ ]:
# Run me first.  Lab 4 needs NOTHING heavy: pure Python + matplotlib/numpy,
# both pre-installed on Colab. No GPU.
import math
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(1337)
np.random.seed(1337)
print("Ready - pure-Python autograd lab, no GPU needed.")


## 1 · Recap - the class you built yesterday

Yesterday you built `Value`: a number that also **remembers how it was computed**.
Every time you combine `Value`s with `+`, `*`, `**`, `exp`, `log`, or `relu`, the
result stores three things:

- `data` - the actual number;
- `_children` - the `Value`s it was built from;
- `_local_grads` - the **local derivative** of the result with respect to each child.

That last one is the key idea. For `c = a * b`, the local derivatives are
$\partial c/\partial a = b$ and $\partial c/\partial b = a$, so the multiply op stores
`(b.data, a.data)`. Each node is a little packet that knows its own one-step calculus.

Run the cell below - **this is exactly the class you built yesterday** (forward only,
no gradients yet).

In [ ]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data ** other, (self,), (other * self.data ** (other - 1),))

    def log(self): return Value(math.log(self.data), (self,), (1 / self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"


A quick forward pass to confirm it still works. Notice every `grad`
is still `0` - we haven't taught it to differentiate yet.

In [ ]:
# A quick forward pass to confirm yesterday's class still works.
a = Value(2.0)
b = Value(-3.0)
c = a * b + a ** 2
print("c      =", c)
print("c.data =", c.data)
print("a.grad =", a.grad, "  <- still 0: we have not run backward() yet")


## 2 · The missing half - `backward()`

Each node currently knows its **local** derivative with respect to its immediate
children. We want the **global** derivative of the final output (the loss) with
respect to *every* node: after `backward()`, `node.grad` should equal
$\partial(\text{output})/\partial(\text{node})$.

The lecture showed how: **walk the graph right-to-left and multiply local derivatives
along the way** (the chain rule). Three ideas make it work.

**(a) The chain rule, one edge at a time.** If the output's gradient at node `v` is
`v.grad`, and `v` was built from `child` with local derivative `local_grad`, then
`child` receives

$$\frac{\partial\,\text{out}}{\partial\,\text{child}} \mathrel{+}=
  \underbrace{local\_grad}_{\partial v / \partial\,child}\times
  \underbrace{v.grad}_{\partial\,\text{out}/\partial v}.$$

That is literally the line `child.grad += local_grad * v.grad`.

**(b) Branches add up - so use `+=`, not `=`.** A value often feeds *several*
downstream nodes (recall the branch slide, where $\partial d/\partial x$ summed the two
paths). Each parent sends back a contribution and they **add**. So we accumulate with
`+=`, and every `grad` must start at `0`.

**(c) Order matters - reverse topological order.** Before we push `v`'s gradient to its
children, `v.grad` must be *finished*: every parent of `v` must already be processed. A
**topological sort** lists nodes so each appears after its children; walking it **in
reverse** guarantees parents-before-children. (This is also why the forward pass *stores*
values - backward *reads* them.)

Putting it together, `backward()`:

1. builds the topological order with a depth-first walk over `_children`;
2. seeds the output with `self.grad = 1` (a node's derivative with respect to itself);
3. walks the order in reverse, doing `child.grad += local_grad * v.grad` on every edge.

### Your turn (1 of 2)
Below is the full class again - identical to yesterday, **plus the new `backward()`
method at the bottom**. Everything is written except the single chain-rule line.
Fill it in. (If a later cell errors with a `TypeError` on `...`, that's this blank -
fill it, or check the Solutions section at the bottom.)

In [ ]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data ** other, (self,), (other * self.data ** (other - 1),))

    def log(self): return Value(math.log(self.data), (self,), (1 / self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ===================== NEW today: the backward() method =====================
    def backward(self):
        # 1) TOPOLOGICAL SORT: list every node so each comes AFTER its children.
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # 2) SEED: the output's gradient with respect to itself is 1.
        self.grad = 1
        # 3) PROPAGATE: walk in REVERSE, pushing each node's grad to its children.
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                # TODO  >> ONE LINE: accumulate the chain-rule contribution.
                #   child's grad should go UP by  (local_grad) x (the grad flowing into v).
                #   Use +=  (a value can feed several parents -> branches ADD up).
                child.grad += ...        # <-- replace the ...


Stuck on the line above? Run this cell - it defines the complete,
correct class so you can carry on. (Try it yourself first!)

## 3 · Sanity check #1 - does it match the lecture?

The lecture worked two examples *by hand*. If our `backward()` is correct, it must
reproduce those exact numbers - the single-neuron case (gradients of $-4$) and the
plenary $y=(3x+2)(x-1)$ at $x=4$ (where $x$ feeds two branches, giving $23$).

In [ ]:
# (1) The single-neuron example from the lecture.
#     x=1, y=2, w=0, b=0   ->   expect  dL/dw = -4  and  dL/db = -4.
x, y = Value(1.0), Value(2.0)
w, b = Value(0.0), Value(0.0)
z = w * x + b           # prediction
L = (z - y) ** 2        # squared-error loss
L.backward()
print(f"L     = {L.data}   (lecture said 4)")
print(f"dL/dw = {w.grad}   (lecture said -4)")
print(f"dL/db = {b.grad}   (lecture said -4)")
assert abs(L.data - 4) < 1e-9
assert abs(w.grad + 4) < 1e-9
assert abs(b.grad + 4) < 1e-9

# (2) The plenary example:  y = (3x + 2)(x - 1)  at x = 4   ->   dy/dx = 23.
x = Value(4.0)
a = 3 * x
bb = a + 2
cc = x - 1
out = bb * cc
out.backward()
print(f"\ny     = {out.data}   (plenary said 42)")
print(f"dy/dx = {x.grad}   (plenary said 23 - note x feeds TWO branches)")
assert abs(out.data - 42) < 1e-9
assert abs(x.grad - 23) < 1e-9
print("\nBoth lecture checks pass - your backward() matches the hand-derivation.")


## 4 · Sanity check #2 - the finite-difference gradient check

Matching the lecture is reassuring, but we want a check that works for **any** function -
even ones we never differentiated by hand. The trick: a derivative *is* a limit of
differences, so we can **approximate** it numerically and compare.

For a small $h$, the **central difference**

$$\frac{\partial f}{\partial x} \approx \frac{f(x+h)-f(x-h)}{2h}$$

is approximately accurate as a rule of thumb. 

If our autograd gradient matches this for several functions,
we can trust it. `grad_check` is the single most useful tool for catching autograd bugs -
professionals use exactly this.

In [ ]:
def grad_check(f, *inputs, h=1e-6, tol=1e-4):
    """Compare Value.backward() gradients to a central finite difference.

    f      : a function of len(inputs) Value args, returning a single Value
    inputs : plain floats - the point at which to check the gradient
    """
    # --- analytic gradient (our autograd) ---
    args = [Value(xi) for xi in inputs]
    f(*args).backward()
    analytic = [arg.grad for arg in args]
    # --- numerical gradient (central difference) ---
    numeric = []
    for i in range(len(inputs)):
        up, dn = list(inputs), list(inputs)
        up[i] += h
        dn[i] -= h
        f_up = f(*[Value(t) for t in up]).data
        f_dn = f(*[Value(t) for t in dn]).data
        numeric.append((f_up - f_dn) / (2 * h))
    # --- compare ---
    ok = True
    for i, (an, nu) in enumerate(zip(analytic, numeric)):
        d = abs(an - nu)
        ok = ok and d < tol
        print(f"  input {i}:  autograd={an:+.6f}   finite-diff={nu:+.6f}   diff={d:.1e}")
    print("PASS\n" if ok else "FAIL\n")
    return ok


print("check 1:  f(a, b) = a*b + a**3 - b/a")
grad_check(lambda a, b: a * b + a ** 3 - b / a, 2.0, -3.0)

print("check 2:  f(a, b, c) = relu(a*b) + exp(c)")
grad_check(lambda a, b, c: (a * b).relu() + c.exp(), 1.5, -2.0, 0.5)


## 5 · A tiny neural network, built on `Value`

The engine is done. A neural net is now just **`Value`s wired together**.

First we need a smooth, differentiable way to turn a raw score into a probability: the
**sigmoid**, $\sigma(v) = 1/(1+e^{-v})$. We add nothing to `Value` - sigmoid is just
`exp`, `+`, and division, so the graph differentiates it for free.

Then three small classes (this is Karpathy's micrograd design):

- **`Neuron`** - a weighted sum $w\cdot x + b$, optionally passed through ReLU;
- **`Layer`** - a row of neurons sharing one input;
- **`MLP`** - a stack of layers; `.parameters()` returns every `Value` to train.

The **last** layer is linear (no ReLU) - we apply `sigmoid` ourselves at the loss.

In [ ]:
def sigmoid(v):
    # 1 / (1 + e^{-v}) - built ENTIRELY from ops Value already has: neg, exp, +, /.
    # Nothing new in Value: the graph (and backward!) handle the derivative for free.
    return 1 / (1 + (-v).exp())

# Even this composed function passes the gradient check:
print("check 3:  f(a, b) = (sigmoid(a*b) - 1)**2")
grad_check(lambda a, b: (sigmoid(a * b) - 1) ** 2, 0.7, -1.3)


In [ ]:
class Neuron:
    """One neuron: output = activation(w . x + b)."""
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    """A row of neurons, all reading the same input."""
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    """A stack of layers. The LAST layer is linear (nonlin=False)."""
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1], nonlin=(i != len(nouts) - 1))
                       for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


Build the network and push a couple of inputs through it. Untrained,
its output is meaningless - but the wiring works, and every parameter is a `Value` ready
to receive a gradient.

In [ ]:
random.seed(1337)
net = MLP(3, [8, 1])          # 3 inputs -> hidden layer of 8 -> 1 output
print("number of parameters:", len(net.parameters()))

# Forward a couple of (made-up) inputs through the untrained net:
print("raw output for [0.2, 0.9, 1.0]:", net([0.2, 0.9, 1.0]))
print("as a probability (sigmoid):    ", sigmoid(net([0.2, 0.9, 1.0])))


## 6 · A tiny dataset - a handful of voters

We reuse Lab 2's synthetic **turnout** data (identical generator). To train *by hand* and
watch every step, we take just **16 rows**. Each voter has three features - age, income,
and whether they voted last time, all scaled to about $[0,1]$ - and a 0/1 label.

In [ ]:
def make_turnout_data(n=2000, seed=0):
    """Synthetic 'did this person vote?' data - the same generator as Lab 2."""
    rng = np.random.default_rng(seed)
    age01 = rng.uniform(0, 1, n)
    income01 = rng.uniform(0, 1, n)
    voted2019 = rng.integers(0, 2, n).astype(float)
    z = (1.6 * age01 + 1.2 * income01 + 2.0 * voted2019
         - 3.0 * age01 * income01 - 1.0)      # interaction term -> a nonlinearity
    p = 1 / (1 + np.exp(-z))
    y = (rng.uniform(0, 1, n) < p).astype(np.int64)
    X = np.stack([age01, income01, voted2019], axis=1).astype(np.float32)
    return X, y


X, y = make_turnout_data(2000, seed=0)

# Take a TINY handful so we can watch every step by hand.
N = 16
Xtr = X[:N].tolist()      # list of [age, income, voted2019]
ytr = y[:N].tolist()      # list of 0/1 labels
print(f"{N} voters, {sum(ytr)} of them voted.")
print("first row:", [round(v, 3) for v in Xtr[0]], "-> label", ytr[0])


## 7 · Train it by hand - forward -> loss -> backward -> update

This is the lecture's "three steps, repeated", written out in full:

1. **Forward + loss.** Push every voter through the net, turn the output into a
   probability with `sigmoid`, and measure the **mean squared error** versus the label.
2. **Zero the grads.** Parameters live *between* steps, so their `.grad` still holds
   *last* step's numbers. Clear them - forgetting this is the #1 beginner bug (the lecture
   warned you).
3. **Backward.** `loss.backward()` fills `.grad` on every parameter.
4. **Update (SGD).** Step each parameter downhill:
   $\theta \leftarrow \theta - \lambda\,\partial L/\partial\theta$.

We record the loss each step so we can plot it - your own version of the lecture's
gradient-descent table.

### Your turn (2 of 2)
Fill in the **SGD update** line - the single step that actually makes the network learn.
The answer is in the Solutions section at the bottom if you need it.

In [ ]:
random.seed(1337)
model = MLP(3, [8, 1])
lr = 1.0
steps = 300
history = []

for step in range(steps):
    # 1) FORWARD + LOSS: mean squared error over all N voters.
    total = Value(0.0)
    for xi, yi in zip(Xtr, ytr):
        pred = sigmoid(model(xi))          # probability in (0, 1)
        total = total + (pred - yi) ** 2   # squared error for this voter
    loss = total * (1.0 / len(Xtr))        # average

    # 2) ZERO THE GRADS (params persist between steps - clear last step's grads!).
    for p in model.parameters():
        p.grad = 0

    # 3) BACKWARD: fill p.grad for every parameter.
    loss.backward()

    # 4) SGD UPDATE: step each parameter downhill.
    for p in model.parameters():
        # TODO  >> ONE LINE: nudge p.data downhill using its gradient.
        #   new value = old value  -  (learning rate) x (gradient)
        p.data -= ...        # <-- replace the ...

    history.append(loss.data)
    if step % 50 == 0 or step == steps - 1:
        print(f"step {step:3d}   loss {loss.data:.4f}")


Stuck on the line above? Run this cell - it defines the complete,
correct class so you can carry on. (Try it yourself first!)

### Watch the loss fall

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history)
plt.xlabel("training step")
plt.ylabel("loss (mean squared error)")
plt.title("Watch it learn: loss vs. training step")
plt.grid(True, alpha=0.3)
plt.show()

print(f"loss went from {history[0]:.4f}  ->  {history[-1]:.4f}")


In [ ]:
correct = 0
for xi, yi in zip(Xtr, ytr):
    prob = sigmoid(model(xi)).data
    pred = 1 if prob > 0.5 else 0
    correct += (pred == yi)
print(f"train accuracy: {correct}/{len(Xtr)} = {correct / len(Xtr):.0%}")
print()
print("(With only 16 points this is partly memorisation - but the ENGINE is real:")
print(" forward -> loss -> backward -> update, exactly like every model you'll train.)")


## 8 · What you just built

That `Value` class is **the whole engine**. Everything in modern deep learning is this
idea at scale:

- **In Lab 11** you'll see Karpathy's pure-Python *microGPT* - a working GPT - run on
  *exactly this `Value` class*, using the same `backward()` you wrote today.
- **From Lab 5 onward**, PyTorch does all of this for you: `loss.backward()` and
  `optimizer.step()` are your `backward()` and your SGD line - just faster, on the GPU,
  and over millions of parameters. **But now you know what they are actually doing.**

You have written reverse-mode automatic differentiation - the single algorithm that
trains every neural network - in under 100 lines.

## 9 · Recap & what's next

Today you:

- added **`backward()`**: topological sort -> seed `self.grad = 1` -> reverse walk with
  `child.grad += local_grad * v.grad`;
- verified it against the lecture **and** with a finite-difference `grad_check`;
- built `Neuron` / `Layer` / `MLP` on `Value` and **trained one by hand**, watching the
  loss fall.

**Lab 5 - the same idea, in PyTorch, at scale.** We'll rebuild today's network in PyTorch
on the *full* turnout dataset: `nn.Module`, an optimiser, and the five-line training loop
from the lecture. Same algorithm - you've just seen the engine room.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — complete Value.backward()**

In [ ]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data ** other, (self,), (other * self.data ** (other - 1),))

    def log(self): return Value(math.log(self.data), (self,), (1 / self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ===================== NEW today: the backward() method =====================
    def backward(self):
        # 1) TOPOLOGICAL SORT: list every node so each comes AFTER its children.
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # 2) SEED: the output's gradient with respect to itself is 1.
        self.grad = 1
        # 3) PROPAGATE: walk in REVERSE, pushing each node's grad to its children.
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

**Solution — the SGD update / full training loop**

In [ ]:
random.seed(1337)
model = MLP(3, [8, 1])
lr = 1.0
steps = 300
history = []

for step in range(steps):
    # 1) FORWARD + LOSS: mean squared error over all N voters.
    total = Value(0.0)
    for xi, yi in zip(Xtr, ytr):
        pred = sigmoid(model(xi))          # probability in (0, 1)
        total = total + (pred - yi) ** 2   # squared error for this voter
    loss = total * (1.0 / len(Xtr))        # average

    # 2) ZERO THE GRADS (params persist between steps - clear last step's grads!).
    for p in model.parameters():
        p.grad = 0

    # 3) BACKWARD: fill p.grad for every parameter.
    loss.backward()

    # 4) SGD UPDATE: step each parameter downhill.
    for p in model.parameters():
        p.data -= lr * p.grad

    history.append(loss.data)
    if step % 50 == 0 or step == steps - 1:
        print(f"step {step:3d}   loss {loss.data:.4f}")